# Shape loss

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from heavyedge import ProfileData

## Profiles with small shape loss

In [ ]:
with ProfileData("local_shape_loss_5p_profiles.h5") as data:
    Ys, _, _ = data[:]
    Ys /= np.sum(Ys, axis=1, keepdims=True)
    lines = plt.plot(np.arange(Ys.shape[1]), Ys.T, color="tab:blue")
    lines[0].set_label(r"$\leq \mathcal{L}_\text{local}^\text{5\%}$")

with ProfileData("shape_loss_5p_profiles.h5") as data:
    Ys, _, _ = data[:]
    Ys /= np.sum(Ys, axis=1, keepdims=True)
    lines = plt.plot(np.arange(Ys.shape[1]), Ys.T, color="tab:orange")
    lines[0].set_label(r"$\leq \mathcal{L}_\text{shape}^\text{5\%}$")

plt.legend()

## Shape loss map

In [ ]:
X = pd.read_csv("X.all_profiles.csv")
shape_loss = pd.read_csv("shape_loss.csv")
Xpred = pd.read_csv("Xpred_2D.csv", index_col=[0, 1])

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("slurry", OneHotEncoder(handle_unknown="ignore"), ["slurry"]),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
)

model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=500,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)
model.fit(X, shape_loss.values.ravel());

In [ ]:
slurries = X["slurry"].unique()

loss_preds = {}
for slurry in slurries:
    xpred = Xpred.copy()
    xpred.insert(0, "slurry", slurry)
    loss_preds[slurry] = model.predict(xpred)

gap = Xpred.groupby(level=0)["gap_to_thickness_ratio"].first().to_numpy()
capillary = Xpred.groupby(level=1)["capillary_number"].first().to_numpy()
vmin = min(loss_pred.min() for loss_pred in loss_preds.values())
vmax = max(loss_pred.max() for loss_pred in loss_preds.values())
levels = np.linspace(vmin, vmax, 21)

fig, axes = plt.subplots(
    2, 2, figsize=(8, 5), sharex=True, sharey=True, constrained_layout=True
)

for ax, slurry in zip(axes.ravel(), slurries):
    loss_pred = loss_preds[slurry]
    loss_map = pd.Series(loss_pred, index=Xpred.index).unstack(level=1).to_numpy()
    contour = ax.contourf(
        gap, capillary, loss_map.T, levels=levels, cmap="jet", vmin=vmin, vmax=vmax
    )
    ax.set_title(slurry)

fig.supxlabel("$R_{gt}$")
fig.supylabel("$Ca$")

fig.colorbar(contour, ax=axes, label="Shape loss")
plt.show()

## Shape loss based quality window

In [ ]:
LOSS_THRESHOLD = 0.4

In [ ]:
slurries = X["slurry"].unique()

loss_preds = {}
for slurry in slurries:
    xpred = Xpred.copy()
    xpred.insert(0, "slurry", slurry)
    loss_preds[slurry] = model.predict(xpred)

gap = Xpred.groupby(level=0)["gap_to_thickness_ratio"].first().to_numpy()
capillary = Xpred.groupby(level=1)["capillary_number"].first().to_numpy()
fig, axes = plt.subplots(
    2, 2, figsize=(8, 5), sharex=True, sharey=True, constrained_layout=True
)

for ax, slurry in zip(axes.ravel(), slurries):
    loss_pred = loss_preds[slurry]
    loss_map = pd.Series(loss_pred, index=Xpred.index).unstack(level=1).to_numpy()
    quality_map = loss_map < LOSS_THRESHOLD
    ax.contourf(
        gap,
        capillary,
        quality_map.T,
        levels=[-0.5, 0.5, 1.5],
        colors=["lightgray", "tab:blue"],
    )
    ax.set_title(slurry)

fig.supxlabel("$R_{gt}$")
fig.supylabel("$Ca$")

fig.legend(
    handles=[
        plt.Rectangle(
            (0, 0), 1, 1, color="tab:blue", label=rf"Predicted loss < {LOSS_THRESHOLD}"
        ),
        plt.Rectangle(
            (0, 0),
            1,
            1,
            color="lightgray",
            label=rf"Predicted loss $\geq$ {LOSS_THRESHOLD}",
        ),
    ],
    loc="lower center",
    ncol=2,
)
plt.show()